### This script will take the folder structure of the results of KSA_project_pipeline 2 and create a single excel file (best_of_all_exps_metrics) with one sheet per experiment to show results of all exps. 

In [7]:
import os
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

# Root folder that contains all experiment folders
ROOT_DIR = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification"

# Filenames to look for inside the "Output" folder, in priority order.
# Both have the same structure (Metric / AvgFolds columns), so whichever exists gets used.
EVAL_METRIC_FILENAMES = [
    "cross_valid_avg_eval_metrics.xlsx",
    "external_validation_eval_metrics.xlsx",
]

# All aggregation method folder names
AGGREGATION_METHODS = [
    "Caption_based_aggregation",
    "Caption_based_aggregation_15_classes",
    "Averaging",
    "Tissue_Type_Clustering"
    # "TITAN",
    # "PRISM"
]

# All model folder names
MODELS = [              
    "H-Optimus-1",
    "Conch1_5",
    "UNI2",
    "Virchow2",
    "ConchV1"
]
# "H-Optimus-1",
# "Conch1_5",
# "UNI2",
# "Virchow2",
# "ConchV1",
# "PRISM"

# Output file path
OUTPUT_FILE = os.path.join(ROOT_DIR, "best_of_all_exps_metric.xlsx")

# ============================================================
# HELPER: extract experiment name from folder name
# ============================================================
def get_experiment_name(folder_name: str) -> str:
    if "_Results" in folder_name:
        return folder_name.split("_Results")[0]
    return folder_name 

# ============================================================
# MAIN
# ============================================================
def build_best_of_all_exps_metric(root_dir, aggregation_methods, models, output_file):

    # Find all experiment folders (folders ending in _Results)
    experiment_folders = [
        f for f in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, f)) and "_Results" in f
    ]

    if not experiment_folders:
        print(f"[WARNING] No experiment folders found in {root_dir}")
        return

    exp_sheets = {}

    for exp_folder in sorted(experiment_folders):
        exp_name = get_experiment_name(exp_folder)
        print(f"\n{'='*60}")
        print(f"Experiment: {exp_name}")
        print(f"{'='*60}")

        agg_blocks = []

        for agg_method in aggregation_methods:
            print(f"  Aggregation: {agg_method}")
            # print(path)
            merged_df = pd.DataFrame()

            for model in models:
                # Define the two specific paths based on your folder structure
                # excel_path = os.path.join(root_dir, exp_folder, agg_method, model, "1-MSIH", "Output", "cross_valid_avg_eval_metrics.xlsx")
                # csv_path = os.path.join(root_dir, exp_folder, agg_method, model, "1-MSIH", "external_validation_PAIP_summary.csv")
                if agg_method == "PRISM" and model != "PRISM":
                    continue
                if agg_method != "PRISM" and model == "PRISM":
                    continue
                
                if agg_method == "PRISM":
                    output_dir = os.path.join(root_dir, exp_folder, "PRISM", "1-MSIH", "Output")
                    csv_path   = os.path.join(root_dir, exp_folder, "PRISM", "1-MSIH", "external_validation_PAIP_summary.csv")
                else:
                    output_dir = os.path.join(root_dir, exp_folder, agg_method, model, "1-MSIH", "Output")
                    csv_path   = os.path.join(root_dir, exp_folder, agg_method, model, "1-MSIH", "external_validation_PAIP_summary.csv")

                # Try each candidate Excel filename in turn — same structure, first match wins
                excel_path = None
                for fname in EVAL_METRIC_FILENAMES:
                    candidate = os.path.join(output_dir, fname)
                    if os.path.exists(candidate):
                        excel_path = candidate
                        break

                # Determine which file to use
                file_path = None
                if excel_path:
                    file_path = excel_path
                elif os.path.exists(csv_path):
                    file_path = csv_path

                if not file_path:
                    continue

                try:
                    # Load based on extension
                    if file_path.endswith('.csv'):
                        df = pd.read_csv(file_path)
                    else:
                        df = pd.read_excel(file_path)

                    if "Metric" not in df.columns:
                        continue

                    # Identify value column (AvgFolds for Excel, Value for CSV)
                    val_col = "AvgFolds" if "AvgFolds" in df.columns else "Value" if "Value" in df.columns else None
                    if not val_col:
                        continue

                    # Standardize data
                    temp = df[["Metric", val_col]].copy()
                    temp["Metric"] = temp["Metric"].astype(str).str.strip()
                    
                    # Capture original row order to prevent sorting issues
                    temp["sort_key"] = range(len(temp))
                    
                    # Append agg suffix to metric name
                    temp["Metric"] = temp["Metric"] + f"_{agg_method}"
                    
                    col_name = f"{model}_N"
                    temp.rename(columns={val_col: col_name}, inplace=True)
                    temp.set_index("Metric", inplace=True)

                    if merged_df.empty:
                        merged_df = temp
                    else:
                        merged_df = merged_df.join(temp[[col_name]], how="outer")
                        # Ensure sort_key is preserved from the first available source
                        if "sort_key" in merged_df.columns:
                            merged_df["sort_key"] = merged_df["sort_key"].fillna(temp["sort_key"])

                except Exception as e:
                    print(f"    [ERROR] {file_path}: {e}")

            if not merged_df.empty:
                # Apply the sort key to restore original classifier order (lin, ann, knn...)
                merged_df = merged_df.sort_values("sort_key").drop(columns=["sort_key"]).reset_index()
                agg_blocks.append(merged_df)
            else:
                print(f"    [SKIP] No data for '{agg_method}'")

        if agg_blocks:
            combined = pd.concat(agg_blocks, axis=0, ignore_index=True)
            exp_sheets[exp_name] = combined
            print(f"  -> Sheet '{exp_name}' created with {len(combined)} rows.")

    # Save to Excel
    if exp_sheets:
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            for sheet_name, df in exp_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"\n✅ All experiments (including External Validation) processed!")
        print(f"Saved to: {output_file}")
    else:
        print("\n[ERROR] No data was collected.")

# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    build_best_of_all_exps_metric(
        ROOT_DIR, 
        AGGREGATION_METHODS, 
        MODELS, 
        OUTPUT_FILE
    )


Experiment: PAIP
  Aggregation: Caption_based_aggregation
  Aggregation: Caption_based_aggregation_15_classes
  Aggregation: Averaging
  Aggregation: Tissue_Type_Clustering
  -> Sheet 'PAIP' created with 120 rows.

Experiment: SurGen
  Aggregation: Caption_based_aggregation
  Aggregation: Caption_based_aggregation_15_classes
    [SKIP] No data for 'Caption_based_aggregation_15_classes'
  Aggregation: Averaging
  Aggregation: Tissue_Type_Clustering
  -> Sheet 'SurGen' created with 90 rows.

Experiment: TCGA_PAIP_EV
  Aggregation: Caption_based_aggregation
  Aggregation: Caption_based_aggregation_15_classes
  Aggregation: Averaging
  Aggregation: Tissue_Type_Clustering
  -> Sheet 'TCGA_PAIP_EV' created with 120 rows.

Experiment: TCGA
  Aggregation: Caption_based_aggregation
  Aggregation: Caption_based_aggregation_15_classes
  Aggregation: Averaging
  Aggregation: Tissue_Type_Clustering
  -> Sheet 'TCGA' created with 120 rows.

✅ All experiments (including External Validation) processe

### Below script will generate radar plots from the best of all exps metrics excel file

In [6]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import os

EVAL_METRIC = "AUC"  # BalAcc or AUC

INPUT_FILE  = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\best_of_all_exps_metric.xlsx"
OUTPUT_DIR  = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\Radar_Plots"

# Aggregation methods — order determines the slice layout (starts from top, clockwise)
AGGREGATION_METHODS = [
    "Caption_based_aggregation",
    # "Caption_based_aggregation_15_classes",
    "Averaging",
    "Tissue_Type_Clustering"
]

# ── Abbreviations used on the radar-plot axis boxes (full names shown in legend key) ──
AGG_ABBREV = {
    "Caption_based_aggregation": "CBA",
    "Caption_based_aggregation_15_classes": "CBA15",
    "Averaging": "AVG",
    "Tissue_Type_Clustering": "TTC"
}

# All model folder names (used to access data)
FOUNDATION_MODELS = ["H-Optimus-1", "Conch1_5", "UNI2", "Virchow2", "ConchV1"]

# ── DISPLAY NAMES ─────────────────────────────────────────────────────────────
# Change the values here to control what appears on the radar plot legend.
# Keys must match FOUNDATION_MODELS exactly; values are the display labels.
MODEL_DISPLAY_NAMES = {
    "H-Optimus-1": "H-Optimus-1",  
    "Conch1_5":    "CONCH1.5",
    "UNI2":        "UNI2",
    "Virchow2":    "Virchow2",
    "ConchV1":     "CONCH",
}
# ──────────────────────────────────────────────────────────────────────────────

MODEL_COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#6b3f00"]

# Background colors (dynamically mapped)
AGGREGATION_COLORS = {
    "Caption_based_aggregation":          "#d0e0ff",
    "Caption_based_aggregation_15_classes": "#fff0b3",  # add a distinct color
    "Averaging":                           "#ffe0e0",
    "Tissue_Type_Clustering":              "#c0ffc0"
}

# Classifier models names on the Radar plot
CLASSIFIER_MODELS = ["Lin", "ANN", "KNN", "Proto", "RF"]
CLASSIFIER_LABEL_MAP = {
    "Lin": "Linear",
    "ANN": "ANN",
    "KNN": "KNN",
    "Proto": "Proto",
    "RF": "RF"
}

MODEL_COLUMN_MAP = {
    "H-Optimus-1": "H-Optimus-1_N",
    "Conch1_5":    "Conch1_5_N",
    "UNI2":        "UNI2_N",
    "Virchow2":    "Virchow2_N",
    "ConchV1":     "ConchV1_N"
}

CLASSIFIER_EXCEL_PREFIX = {
    "Lin": "lin",
    "ANN": "ann",
    "KNN": "knn",
    "Proto": "proto",
    "RF": "rf"
}

EVAL_METRIC_EXCEL_SUFFIX = {
    "BalAcc": "bacc",
    "AUC": "auroc",
    "Acc": "acc",
    "MacroF1": "macro_f1",
    "WF1": "weighted_f1",
}

# ============================================================
# HELPERS
# ============================================================

def close_circle(data_list):
    return data_list + [data_list[0]]

def extract_values(df, experiment_name, aggregation_methods, classifier_models,
                   eval_metric, foundation_models, model_column_map, agg_abbrev):
    df["Metric"] = df["Metric"].astype(str).str.strip()
    axis_labels, agg_groups = [], []
    models_data = {fm: [] for fm in foundation_models}

    for agg in aggregation_methods:
        agg_short = agg_abbrev.get(agg, agg)
        for clf in classifier_models:
            full_clf = CLASSIFIER_LABEL_MAP.get(clf, clf)
            axis_labels.append(f"{full_clf}-{eval_metric}\n({agg_short})")
            agg_groups.append(agg)  # keep full name here for color/legend lookups

            clf_prefix = CLASSIFIER_EXCEL_PREFIX.get(clf, clf.lower())
            metric_suffix = EVAL_METRIC_EXCEL_SUFFIX.get(eval_metric, eval_metric.lower())
            metric_row = f"{clf_prefix}_{metric_suffix}_{agg}"
            row = df[df["Metric"] == metric_row]

            for fm in foundation_models:
                col = model_column_map[fm]
                if row.empty or col not in df.columns:
                    models_data[fm].append(0.0)
                else:
                    val = row[col].iloc[0]
                    models_data[fm].append(round(float(val), 4) if pd.notna(val) else 0.0)

    return axis_labels, agg_groups, models_data

# ============================================================
# MAIN
# ============================================================

def generate_radar_plots(input_file, output_dir, aggregation_methods, aggregation_colors,
                         classifier_models, foundation_models, model_colors,
                         model_column_map, eval_metric, model_display_names, agg_abbrev):
    os.makedirs(output_dir, exist_ok=True)
    xl = pd.ExcelFile(input_file)

    for exp_name in xl.sheet_names:
        df = pd.read_excel(input_file, sheet_name=exp_name)

        axis_labels, agg_groups, models_data = extract_values(
            df, exp_name, aggregation_methods, classifier_models,
            eval_metric, foundation_models, model_column_map, agg_abbrev
        )

        # ── 1. Fixed max radius — always the full 0-1.0 circle ──
        max_r = 1.0
        label_position = 1.13
        title_y_position = 1.12

        N = len(axis_labels)
        angle_step = 2 * np.pi / N
        half_step  = angle_step / 2
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False) + half_step
        angles_closed = np.concatenate((angles, [angles[0]]))

        fig = plt.figure(figsize=(24, 24))
        ax  = fig.add_subplot(111, polar=True)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # ── Background fills ───────────────────────────────────
        n_groups = len(aggregation_methods)
        group_span = 2 * np.pi / n_groups

        for i, agg in enumerate(aggregation_methods):
            theta_start = i * group_span
            theta_end   = (i + 1) * group_span
            theta_arc   = np.linspace(theta_start, theta_end, 200)
            theta_wedge = np.concatenate(([theta_start], theta_arc, [theta_end]))
            r_wedge     = np.concatenate(([0], np.full(len(theta_arc), max_r), [0]))
            ax.fill(theta_wedge, r_wedge, color=aggregation_colors.get(agg, "#f0f0f0"), alpha=0.4, zorder=0)

        # ── Plot each foundation model (using display names for legend) ──
        for (fm, values), color in zip(models_data.items(), model_colors):
            display_name = model_display_names.get(fm, fm)  # fall back to key if not mapped
            stats = close_circle(values)
            ax.plot(angles_closed, stats, color=color, label=display_name,
                    linewidth=3, marker='o', markersize=6, zorder=5)
            ax.fill(angles_closed, stats, color=color, alpha=0.06, zorder=4)

        # ── Axis tick labels ───────────────────────────────────
        ax.set_xticks(angles)
        ax.set_xticklabels([])

        # clip_on=False stops these boxes from being cut off at the plot boundary
        for angle, label, agg in zip(angles, axis_labels, agg_groups):
            ax.text(angle, label_position, label, ha="center", va="center",
                    fontsize=14, fontweight="bold", rotation=0, clip_on=False,
                    bbox=dict(facecolor=aggregation_colors.get(agg, "#f0f0f0"),
                              edgecolor="black", boxstyle="square,pad=0.5", linewidth=1.5))

        # ── Scale numbers along every spoke (fixed grid: 0.10 ... 0.90) ──
        ax.set_yticklabels([])
        ax.set_ylim(0.0, max_r)
        r_ticks = np.arange(0.50, max_r, 0.10)
        ax.set_yticks(r_ticks)

        for angle in angles:
            for r in r_ticks:
                ax.text(angle, r, f"{r:.2f}", ha="center", va="center",
                        fontsize=24, color="black", zorder=6, clip_on=False)

        # ── Legends ────────────────────────────────────────────
        model_handles, model_labels = ax.get_legend_handles_labels()
        fig.legend(model_handles, model_labels, loc="upper right", bbox_to_anchor=(0.98, 1.02),
                   title="Models", fontsize=18, title_fontsize=20, borderpad=1.2,
                   framealpha=0.95, edgecolor="black", fancybox=True, shadow=True)

        # Legend now spells out what each abbreviation on the radar axes means
        agg_handles = [mpatches.Patch(facecolor=aggregation_colors.get(agg, "#f0f0f0"),
                       edgecolor="black", label=f"{agg_abbrev.get(agg, agg)} = {agg}")
                       for agg in aggregation_methods]
        fig.legend(handles=agg_handles, loc="upper left", bbox_to_anchor=(-0.02, 1.02),
                   title="Aggregation Methods (Key)", fontsize=16, title_fontsize=18,
                   borderpad=1.2, framealpha=0.95, edgecolor="black", fancybox=True, shadow=True)

        plt.title(f"Radar Plot — {exp_name} | Metric: {eval_metric}",
                  y=title_y_position, fontsize=22, fontweight="bold")

        save_path = os.path.join(output_dir, f"{exp_name}_{eval_metric}_radar.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.5)
        plt.close()
        print(f"  ✅ Saved: {save_path}")


if __name__ == "__main__":
    generate_radar_plots(
        INPUT_FILE, OUTPUT_DIR, AGGREGATION_METHODS, AGGREGATION_COLORS,
        CLASSIFIER_MODELS, FOUNDATION_MODELS, MODEL_COLORS,
        MODEL_COLUMN_MAP, EVAL_METRIC,
        MODEL_DISPLAY_NAMES, AGG_ABBREV
    )

  ✅ Saved: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\Radar_Plots\PAIP_AUC_radar.png
  ✅ Saved: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\Radar_Plots\SurGen_AUC_radar.png
  ✅ Saved: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\Radar_Plots\TCGA_PAIP_EV_AUC_radar.png
  ✅ Saved: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\Radar_Plots\TCGA_AUC_radar.png


In [6]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import os

# ============================================================
# CONFIG — change these to control what is plotted
# ============================================================

EVAL_METRIC = "AUC"   # BalAcc | AUC | Acc | MacroF1 | WF1

INPUT_FILE = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\best_of_all_exps_metric.xlsx"
OUTPUT_DIR = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\TITAN_vs_PRISM"

# TITAN data lives under Conch1_5_N (model-agnostic, same as Radar script)
# PRISM data lives under PRISM_N
TITAN_COL = "Conch1_5_N"
PRISM_COL = "PRISM_N"

# Classifiers shown as axes on the plot
CLASSIFIER_MODELS = ["Lin", "ANN", "KNN", "Proto", "RF"]
CLASSIFIER_LABEL_MAP = {
    "Lin":   "Linear",
    "ANN":   "ANN",
    "KNN":   "KNN",
    "Proto": "Proto",
    "RF":    "RF",
}
CLASSIFIER_EXCEL_PREFIX = {
    "Lin":   "lin",
    "ANN":   "ann",
    "KNN":   "knn",
    "Proto": "proto",
    "RF":    "rf",
}

EVAL_METRIC_EXCEL_SUFFIX = {
    "BalAcc":  "bacc",
    "AUC":     "auroc",
    "Acc":     "acc",
    "MacroF1": "macro_f1",
    "WF1":     "weighted_f1",
}

# Visual identity for the two models
TITAN_COLOR = "#4363d8"   # blue
PRISM_COLOR = "#e6194b"   # red

# Background wedge colours
WEDGE_COLORS = {
    "TITAN": "#dce8ff",
    "PRISM": "#ffe0e0",
}

# ============================================================
# HELPERS
# ============================================================

def close_circle(data_list):
    return data_list + [data_list[0]]

def get_value(df, clf_prefix, metric_suffix, agg_tag, col):
    key = f"{clf_prefix}_{metric_suffix}_{agg_tag}"
    row = df[df["Metric"] == key]
    if row.empty or col not in df.columns:
        return 0.0
    val = row[col].iloc[0]
    return round(float(val), 4) if pd.notna(val) else 0.0

def extract_values(df, eval_metric):
    metric_suffix = EVAL_METRIC_EXCEL_SUFFIX.get(eval_metric, eval_metric.lower())
    axis_labels = []
    models_data = {"TITAN": [], "PRISM": []}

    for clf in CLASSIFIER_MODELS:
        full_clf   = CLASSIFIER_LABEL_MAP.get(clf, clf)
        clf_prefix = CLASSIFIER_EXCEL_PREFIX.get(clf, clf.lower())
        axis_labels.append(f"{full_clf}\n({eval_metric})")
        models_data["TITAN"].append(get_value(df, clf_prefix, metric_suffix, "TITAN", TITAN_COL))
        models_data["PRISM"].append(get_value(df, clf_prefix, metric_suffix, "PRISM", PRISM_COL))

    return axis_labels, models_data

# ============================================================
# MAIN PLOT
# ============================================================

def generate_comparison_plots(input_file, output_dir, eval_metric):
    os.makedirs(output_dir, exist_ok=True)
    xl = pd.ExcelFile(input_file)

    for exp_name in xl.sheet_names:
        df = pd.read_excel(input_file, sheet_name=exp_name)
        df["Metric"] = df["Metric"].astype(str).str.strip()

        axis_labels, models_data = extract_values(df, eval_metric)

        # ── Fixed radius — always the full 0-1.0 circle ────────
        max_r = 1.0
        label_position = 1.13
        title_y_pos    = 1.12

        N             = len(axis_labels)
        half_step     = (2 * np.pi / N) / 2
        angles        = np.linspace(0, 2 * np.pi, N, endpoint=False) + half_step
        angles_closed = np.concatenate((angles, [angles[0]]))

        fig = plt.figure(figsize=(24, 24))
        ax  = fig.add_subplot(111, polar=True)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # ── Background: left half TITAN, right half PRISM ──────
        r_wedge = np.concatenate(([0], np.full(300, max_r), [0]))

        arc_t   = np.linspace(half_step, half_step + np.pi, 300)
        ax.fill(np.concatenate(([half_step], arc_t, [half_step + np.pi])),
                r_wedge, color=WEDGE_COLORS["TITAN"], alpha=0.45, zorder=0)

        arc_p   = np.linspace(half_step + np.pi, half_step + 2 * np.pi, 300)
        ax.fill(np.concatenate(([half_step + np.pi], arc_p, [half_step + 2 * np.pi])),
                r_wedge, color=WEDGE_COLORS["PRISM"], alpha=0.45, zorder=0)

        # ── Plot TITAN and PRISM ───────────────────────────────
        plot_cfg = [
            ("TITAN", TITAN_COLOR, "-",  "o"),
            ("PRISM", PRISM_COLOR, "--", "s"),
        ]
        for model_name, color, ls, marker in plot_cfg:
            values = models_data[model_name]
            stats  = close_circle(values)
            ax.plot(angles_closed, stats,
                    color=color, label=model_name,
                    linewidth=3.5, linestyle=ls,
                    marker=marker, markersize=8, zorder=5)
            ax.fill(angles_closed, stats, color=color, alpha=0.08, zorder=4)

            # Value label at each spoke
            # for angle, val in zip(angles, values):
            #     if val > 0:
            #         ax.text(angle, val + 0.025, f"{val:.3f}",
            #                 ha="center", va="bottom",
            #                 fontsize=11, fontweight="bold", color=color, zorder=7)

        # ── Spoke labels (same style as radar script) ──────────
        ax.set_xticks(angles)
        ax.set_xticklabels([])

        for angle, label in zip(angles, axis_labels):
            norm_angle = (angle - half_step) % (2 * np.pi)
            bg = WEDGE_COLORS["TITAN"] if norm_angle < np.pi else WEDGE_COLORS["PRISM"]
            ax.text(angle, label_position, label,
                    ha="center", va="center",
                    fontsize=14, fontweight="bold", rotation=0,
                    bbox=dict(facecolor=bg, edgecolor="black",
                              boxstyle="square,pad=0.5", linewidth=1.5))

        # ── Radial scale (fixed grid: 0.10 ... 0.90) ───────────
        ax.set_yticklabels([])
        ax.set_ylim(0.0, max_r)
        r_ticks = np.arange(0.50, max_r, 0.10)
        ax.set_yticks(r_ticks)

        for angle in angles:
            for r in r_ticks:
                ax.text(angle, r, f"{r:.2f}",
                        ha="center", va="center",
                        fontsize=16, color="black", fontweight="bold", zorder=6)

        # ── Legends (same positions as radar script) ───────────
        model_handles, model_labels_leg = ax.get_legend_handles_labels()
        fig.legend(model_handles, model_labels_leg,
                   loc="upper right", bbox_to_anchor=(0.98, 0.98),
                   title="Aggregation Model", fontsize=18, title_fontsize=20,
                   borderpad=1.2, framealpha=0.98,
                   edgecolor="black", fancybox=True, shadow=True)

        wedge_handles = [
            mpatches.Patch(facecolor=WEDGE_COLORS["TITAN"], edgecolor="black", label="TITAN region"),
            mpatches.Patch(facecolor=WEDGE_COLORS["PRISM"], edgecolor="black", label="PRISM region"),
        ]
        fig.legend(handles=wedge_handles,
                   loc="upper left", bbox_to_anchor=(0.02, 0.98),
                   title="Background", fontsize=18, title_fontsize=20,
                   borderpad=1.2, framealpha=0.98,
                   edgecolor="black", fancybox=True, shadow=True)

        plt.title(f"TITAN  vs  PRISM  —  {exp_name}  |  Metric: {eval_metric}",
                  y=title_y_pos, fontsize=22, fontweight="bold")

        save_path = os.path.join(output_dir, f"{exp_name}_{eval_metric}_TITAN_vs_PRISM.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.5)
        plt.close()
        print(f"  ✅ Saved: {save_path}")


# ============================================================
# RUN
# ============================================================
generate_comparison_plots(INPUT_FILE, OUTPUT_DIR, EVAL_METRIC)

  ✅ Saved: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Analysis_and_Visualization\TITAN_vs_PRISM\SurGen_AUC_TITAN_vs_PRISM.png
